# Mi restaurante de sushi: predicción de demanda, facturación y rentabilidad

**Mis tres objetivos** son:

1. Predecir **pedidos diarios** mediante machine learning.
2. Predecir **facturación diaria** de dos maneras: con machine learning y con `pedidos previstos × ticket medio`.
3. Calcular un **beneficio operativo estimado** restando costes variables, personal y costes fijos.

Trabajo con `sushi_orders.csv` y `sushi_daily_pnl.csv` del ZIP facilitado. Son datos de **2015 a 2024**.

> Para ejecutar este notebook, abrirlo desde la carpeta del proyecto (la misma donde están `backend.py` y `data/`).

## 1. Importar librerías
Empiezo importando las librerías necesarias para leer archivos, analizar datos y crear gráficos.

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from backend import (
    cargar_datos, renombrar_columnas, resumen_diario, preparar_variables, separar_temporal,
    medias_historicas, COLUMNAS_DEMANDA, COLUMNAS_FACTURACION,
    cargar_modelo, predecir_futuro, calcular_rentabilidad, punto_equilibrio
)
from train_models import crear_modelos, main as entrenar_todos_los_modelos


## 2. Leer mis dos archivos CSV
El archivo de pedidos contiene **una fila por pedido**. El de rentabilidad contiene **una fila por día**, con costes y beneficio operativo.

In [2]:
pedidos_df, gastos_df = cargar_datos()
print("Pedidos:", pedidos_df.shape)
print("Gastos:", gastos_df.shape)
display(pedidos_df.head())
display(gastos_df.head())

Pedidos: (83890, 21)
Gastos: (3653, 15)


,order_id,date,year,month,quarter,day_of_week,is_weekend,is_holiday,customer_id,customer_segment,...,channel,party_size,revenue,cogs,gross_profit,discount,tip,delivery_fee,total_collected,rating
0,1,2015-01-01,2015,1,1,Thu,False,True,79,occasional,...,dine_in,2,29.0,9.14,19.86,0.0,5.47,0.0,34.47,NaN
1,2,2015-01-01,2015,1,1,Thu,False,True,50,occasional,...,takeaway,3,86.5,26.92,59.58,0.0,0.00,0.0,86.50,NaN
2,3,2015-01-01,2015,1,1,Thu,False,True,44,one_time,...,dine_in,2,53.5,14.61,38.89,0.0,7.86,0.0,61.36,NaN
3,4,2015-01-01,2015,1,1,Thu,False,True,27,regular,...,takeaway,5,180.5,53.37,127.13,0.0,0.00,0.0,180.50,NaN
4,5,2015-01-01,2015,1,1,Thu,False,True,4,occasional,...,dine_in,4,162.0,52.04,109.96,16.2,42.74,0.0,188.54,NaN


,date,year,month,quarter,day_of_week,is_weekend,is_holiday,covers,gross_revenue,cogs,gross_profit,labour_cost,fixed_ops_cost,operating_profit,gp_margin_pct
0,2015-01-01,2015,1,1,Thu,False,True,21,925.0,282.01,642.99,626.85,160.33,-144.19,69.5
1,2015-01-02,2015,1,1,Fri,False,False,19,716.0,237.08,478.92,626.85,160.87,-308.80,66.9
2,2015-01-03,2015,1,1,Sat,True,False,34,1315.5,407.74,907.76,671.85,174.28,61.63,69.0
3,2015-01-04,2015,1,1,Sun,True,False,33,1229.0,388.54,840.46,671.85,162.53,6.08,68.4
4,2015-01-05,2015,1,1,Mon,False,False,12,626.0,199.44,426.56,601.85,164.31,-339.60,68.1


## 3. Comprobar que los datos tienen sentido
Antes de modelar, compruebo fechas, duplicados y valores nulos.

In [3]:
print("Pedidos repetidos por ID:", pedidos_df["order_id"].duplicated().sum())
print("Días repetidos en Gastos:", gastos_df["date"].duplicated().sum())
print("Fechas:", pedidos_df["date"].min().date(), "a", pedidos_df["date"].max().date())
print("Nulos que sí debo revisar:")
print(pedidos_df[["order_id", "date", "revenue", "cogs"]].isna().sum())
print("¿Coinciden los ingresos de ambos archivos?:",
      np.isclose(pedidos_df["revenue"].sum(), gastos_df["gross_revenue"].sum()))

Pedidos repetidos por ID: 0
Días repetidos en Gastos: 0
Fechas: 2015-01-01 a 2024-12-31
Nulos que sí debo revisar:
order_id    0
date        0
revenue     0
cogs        0
dtype: int64
¿Coinciden los ingresos de ambos archivos?: True


### 3.1 Cambiar nombre de columnas

In [4]:
pedidos_df = renombrar_columnas(gastos_df)

pedidos_df.head()

,fecha,anio,mes,trimestre,dia_semana,es_fin_de_semana,es_festivo,covers,gross_revenue,coste_productos_vendidos,beneficio_bruto,labour_cost,fixed_ops_cost,operating_profit,gp_margin_pct
0,2015-01-01,2015,1,1,Thu,False,True,21,925.0,282.01,642.99,626.85,160.33,-144.19,69.5
1,2015-01-02,2015,1,1,Fri,False,False,19,716.0,237.08,478.92,626.85,160.87,-308.80,66.9
2,2015-01-03,2015,1,1,Sat,True,False,34,1315.5,407.74,907.76,671.85,174.28,61.63,69.0
3,2015-01-04,2015,1,1,Sun,True,False,33,1229.0,388.54,840.46,671.85,162.53,6.08,68.4
4,2015-01-05,2015,1,1,Mon,False,False,12,626.0,199.44,426.56,601.85,164.31,-339.60,68.1


In [5]:
gastos_df = renombrar_columnas(gastos_df)

gastos_df.head()

,fecha,anio,mes,trimestre,dia_semana,es_fin_de_semana,es_festivo,covers,gross_revenue,coste_productos_vendidos,beneficio_bruto,labour_cost,fixed_ops_cost,operating_profit,gp_margin_pct
0,2015-01-01,2015,1,1,Thu,False,True,21,925.0,282.01,642.99,626.85,160.33,-144.19,69.5
1,2015-01-02,2015,1,1,Fri,False,False,19,716.0,237.08,478.92,626.85,160.87,-308.80,66.9
2,2015-01-03,2015,1,1,Sat,True,False,34,1315.5,407.74,907.76,671.85,174.28,61.63,69.0
3,2015-01-04,2015,1,1,Sun,True,False,33,1229.0,388.54,840.46,671.85,162.53,6.08,68.4
4,2015-01-05,2015,1,1,Mon,False,False,12,626.0,199.44,426.56,601.85,164.31,-339.60,68.1


## 4. Crear mi DataFrame diario
Agrupo por fecha para obtener la **variable objetivo de demanda**: número de pedidos únicos. También calculo la facturación total del día y uno los costes del segundo CSV.

In [6]:
diario_df = resumen_diario(pedidos_df, gastos_df)
display(diario_df.head(10))
print("Días totales:", len(diario_df))
print("Pedidos:", diario_df["pedidos_diarios"].sum())
print("Facturación:", round(diario_df["facturacion_diaria"].sum(), 2))

KeyError: "Label(s) ['id_pedido', 'ingresos'] do not exist"

## 5. Análisis exploratorio (EDA)
Antes de predecir me interesa conocer **cómo cambian los pedidos y los ingresos con el tiempo**.

In [ ]:
por_año = diario_df.assign(anio=diario_df["date"].dt.year).groupby("anio").agg(
    pedidos=("pedidos_diarios", "sum"),
    facturacion=("facturacion_diaria", "sum"),
    beneficio=("operating_profit", "sum")
)
display(por_año.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
por_año["pedidos"].plot(kind="bar", ax=axes[0], title="Pedidos por año")
por_año["facturacion"].plot(kind="bar", ax=axes[1], title="Facturación por año")
axes[0].set_ylabel("Pedidos")
axes[1].set_ylabel("Importe")
plt.tight_layout()
plt.show()

### Variación por día de la semana
Busco días de mayor y menor actividad para entender el comportamiento de la demanda.

In [ ]:
dias_es = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
promedio_dia = diario_df.assign(dia=diario_df["date"].dt.dayofweek).groupby("dia")["pedidos_diarios"].mean()
promedio_dia.index = dias_es
plt.figure(figsize=(10, 4))
promedio_dia.plot(kind="bar")
plt.title("Promedio de pedidos por día de la semana")
plt.ylabel("Pedidos medios por jornada")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### Evolución histórica y canales de venta
Reviso también la tendencia de 30 días y de dónde proceden los pedidos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ventas_serie = diario_df.set_index("date")["pedidos_diarios"].rolling(30, min_periods=1).mean()
ventas_serie.plot(ax=axes[0], title="Media móvil de pedidos (30 días)")
pedidos_df["channel"].value_counts().plot(kind="bar", ax=axes[1], title="Pedidos por canal")
axes[0].set_ylabel("Pedidos medios")
axes[1].set_ylabel("Total pedidos")
plt.tight_layout()
plt.show()

## 6. Crear las variables de machine learning
Añado variables de calendario y las **ventas o pedidos de días anteriores**, nunca del mismo día que voy a predecir. El `.shift(1)` evita usar información futura en cada fila.

In [ ]:
df_ml = preparar_variables(diario_df)
display(df_ml[["date"] + COLUMNAS_DEMANDA + ["pedidos_diarios"]].head(10))
print("Total de días utilizables:", len(df_ml))

## 7. Separar entrenamiento y prueba de manera cronológica
No uso `train_test_split(shuffle=True)` porque mezclando fechas aprendería del futuro. Reservo el último 20 % de jornadas para comprobar mis modelos.

In [ ]:
train_df, test_df = separar_temporal(df_ml)
X_train_demanda = train_df[COLUMNAS_DEMANDA]
X_test_demanda = test_df[COLUMNAS_DEMANDA]
y_train_demanda = train_df["pedidos_diarios"]
y_test_demanda = test_df["pedidos_diarios"]
print("Entrenamiento:", X_train_demanda.shape)
print("Prueba:", X_test_demanda.shape)
print("Fecha inicial de prueba:", test_df["date"].min().date())

## 8. Repasar los nueve modelos de regresión
Son los mismos modelos que utilicé en mi notebook anterior. KNN y SVR incorporan `StandardScaler` porque necesitan comparar las variables en escalas similares.

In [ ]:
modelos = crear_modelos()
print("Modelos que voy a comparar:")
for nombre in modelos:
    print("-", nombre)

### Ejemplo: entrenar un modelo y evaluar MAE, RMSE y R²
Para entender el proceso, primero entreno **Random Forest** y lo evalúo con los días de prueba. Después entrenaré todos con la misma lógica.

In [ ]:
mi_random_forest = crear_modelos()["Random Forest"]
mi_random_forest.fit(X_train_demanda, y_train_demanda)
pred_rf = np.maximum(mi_random_forest.predict(X_test_demanda), 0)
print("MAE:", round(mean_absolute_error(y_test_demanda, pred_rf), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_demanda, pred_rf)), 3))
print("R²:", round(r2_score(y_test_demanda, pred_rf), 3))

## 9. Entrenar los 9 modelos de demanda y los 9 de facturación
Esta función recorre los nueve regresores para **cada una de mis dos variables objetivo**. Guarda modelos `.joblib`, tablas de comparación y predicciones de prueba en `modelos/`. Esto permite que Streamlit funcione sin entrenar cada vez que abro la web.

In [ ]:
entrenar_todos_los_modelos()

## 10. Comparar los resultados
Ordeno cada tabla por **menor MAE**. La demanda tiene MAE en pedidos; la facturación tiene MAE en las unidades monetarias del dataset.

In [ ]:
from backend import MODELOS
resultados_demanda = pd.read_csv(MODELOS / "comparacion_demanda.csv")
resultados_facturacion = pd.read_csv(MODELOS / "comparacion_facturacion.csv")
print("Demanda:")
display(resultados_demanda.round(3))
print("Facturación:")
display(resultados_facturacion.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
resultados_demanda.sort_values("MAE").plot.barh(x="Modelo", y="MAE", ax=axes[0], legend=False, title="MAE de demanda")
resultados_facturacion.sort_values("MAE").plot.barh(x="Modelo", y="MAE", ax=axes[1], legend=False, title="MAE de facturación")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Cargar los resultados de los modelos
demanda = pd.read_csv("modelos/comparacion_demanda.csv")
facturacion = pd.read_csv("modelos/comparacion_facturacion.csv")

# 2. Unir los resultados por modelo
comparacion = demanda[["Modelo", "R2"]].merge(
    facturacion[["Modelo", "R2"]],
    on="Modelo",
    suffixes=("_Demanda", "_Facturacion")
)

# 3. Crear el gráfico de barras
x = np.arange(len(comparacion))
ancho = 0.35

plt.figure(figsize=(13, 6))

plt.bar(
    x - ancho/2,
    comparacion["R2_Demanda"],
    width=ancho,
    label="Demanda"
)

plt.bar(
    x + ancho/2,
    comparacion["R2_Facturacion"],
    width=ancho,
    label="Facturación"
)

# 4. Personalizar el gráfico
plt.xticks(x, comparacion["Modelo"], rotation=45, ha="right")
plt.ylabel("R²")
plt.xlabel("Modelos de Machine Learning")
plt.title("Comparación del R²: Demanda vs Facturación")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Comparar la predicción con la realidad
Los gráficos muestran predicciones a **un día vista** sobre el conjunto de prueba. Para el pronóstico de varios días sin observaciones reales, usaré una predicción recursiva y debo ser más prudente.

In [ ]:
mejor_demanda = resultados_demanda.iloc[0]["Modelo"]
mejor_facturacion = resultados_facturacion.iloc[0]["Modelo"]
prueba_demanda = pd.read_csv(MODELOS / "predicciones_test_demanda.csv", parse_dates=["date"])
prueba_facturacion = pd.read_csv(MODELOS / "predicciones_test_facturacion.csv", parse_dates=["date"])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for ax, tabla, modelo, titulo in [
    (axes[0], prueba_demanda, mejor_demanda, "Pedidos"),
    (axes[1], prueba_facturacion, mejor_facturacion, "Facturación"),
]:
    muestra = tabla.head(60)
    ax.plot(muestra["date"], muestra["real"], label="Real")
    ax.plot(muestra["date"], muestra[modelo], label=f"Previsto: {modelo}")
    ax.set_title(titulo + " — 60 primeros días de prueba")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Mi segunda métrica: predicción de facturación
Mantengo la fórmula de mi primer proyecto: `pedidos previstos × ticket medio`. Además, en `train_models.py` entreno nueve regresores directamente para **facturacion_diaria**. Puedo comparar ambos resultados, aunque utilizan información distinta.

In [ ]:
referencias = medias_historicas(pedidos_df, gastos_df)
ticket = referencias["ticket_medio"]  # media del último año del dataset (2024)
print("Ticket medio del último año:", round(ticket, 2))
print("Mejor modelo directo de facturación:", mejor_facturacion)

modelo_pedidos = cargar_modelo("demanda", mejor_demanda)
modelo_ventas = cargar_modelo("facturacion", mejor_facturacion)
previsiones = predecir_futuro(diario_df, modelo_pedidos, modelo_ventas, dias=14)
previsiones["facturacion_por_ticket"] = previsiones["pedidos_previstos"] * ticket
display(previsiones.round(2))

### Gráfico de facturación prevista
Las dos líneas son pronósticos distintos, no ventas reales. Por defecto, la previsión empieza al terminar el histórico, es decir, el 1 de enero de 2025.

In [ ]:
previsiones.set_index("date")[["facturacion_ml", "facturacion_por_ticket"]].plot(
    figsize=(12, 4), marker="o", title="Comparación de dos previsiones de facturación")
plt.ylabel("Importe estimado")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Mi tercera métrica: estimación de rentabilidad
El segundo CSV trae costes reales de ingredientes (`cogs`), personal (`labour_cost`) y costes fijos (`fixed_ops_cost`). Para días futuros no conozco esos importes: uso **supuestos modificables**. No presento el beneficio calculado como un tercer modelo predictivo independiente.

**Fórmula:** beneficio operativo = facturación prevista − ingredientes − personal − costes fijos. No incluye impuestos ni inversiones iniciales.

In [ ]:
costes_variables_pct = referencias["porcentaje_coste_variable"]
personal_mensual = referencias["personal_diario"] * 30
costes_fijos_mensuales = referencias["costes_fijos_diarios"] * 30

economia = calcular_rentabilidad(
    previsiones, ticket_medio=ticket,
    porcentaje_coste_variable=costes_variables_pct,
    personal_mensual=personal_mensual,
    costes_fijos_mensuales=costes_fijos_mensuales,
    dias_apertura_mes=30,
    usar_facturacion_ml=False  # uso pedidos previstos × ticket medio
)
display(economia[["date", "pedidos_previstos", "facturacion_prevista", "costes_variables",
                  "costes_personal", "costes_fijos", "beneficio_operativo"]].round(2))
print("Beneficio operativo total previsto:", round(economia["beneficio_operativo"].sum(), 2))

### Gráfico de beneficio previsto y punto de equilibrio

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(economia["date"], economia["beneficio_operativo"])
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Beneficio operativo previsto por día")
ax.set_ylabel("Importe")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

pedidos_minimos = punto_equilibrio(ticket, costes_variables_pct,
                                  personal_mensual, costes_fijos_mensuales, 30)
print("Pedidos diarios mínimos para cubrir costes:", pedidos_minimos)

## 14. Conclusiones y uso de Streamlit
He realizado un análisis completo con **dos modelos de predicción** (demanda y facturación) y **un cálculo de rentabilidad basado en costes**.

**Para usar la aplicación:** abrir una terminal en la carpeta del proyecto y ejecutar `python -m streamlit run app.py`. Desde la barra lateral puedo elegir el modelo, el ticket medio, los costes y cuántos días quiero proyectar.

**Límites:** estos datos corresponden a un restaurante de sushi, no a mi futuro restaurante africano; terminan en 2024, por lo que no validan una predicción real de 2026. Las métricas del 20 % de prueba son de un día vista, no de 14 o 30 días recursivos. No tengo información horaria.